# Early mistakes and recorded MATHStream completion

**Dasey Dang · Carnegie Learning capstone · Companion analysis**

**Research question.** Among attempts that reach three distinct recorded activities, are early errors associated with recorded completion?

This analysis uses the first three distinct activities of any item type, following the audited completion workflow. The outcome is the status in the attempt export. See `04_early_errors_assessment.ipynb` for the main analysis of later assessment errors.

Run top to bottom from a fresh Python kernel. The descriptive analysis runs in Python. Set `run_r_models = True` for the mixed-effects completion model and the held-out prediction evaluation. See `MATHStream_Reproducibility.md` for setup. The notebook saves aggregate outputs to `results/early_errors_completion/`.

## 1. Load exports and order distinct activities

Use the earliest recorded response per student, attempt and activity ID. See the codebook for the join key, timestamp handling and response codes.

In [ ]:
import os
from pathlib import Path
from textwrap import fill

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

# Set MATHSTREAM_DATA_DIR to use a private input directory outside the repository.
cwd = Path.cwd()
root = cwd.parent if cwd.name in {"notebooks", "02_notebook"} else cwd
data_dir = Path(os.environ.get("MATHSTREAM_DATA_DIR", root / "data" / "raw")).expanduser()
results = root / "results" / "early_errors_completion"
keys = ["anon_userid", "attemptid"]
scored_states = ["CORRECT", "INCORRECT"]

run_r_models = False
split_seed = 2026
test_student_fraction = 0.20

In [ ]:
attempt_fields = ["status", "lessonstreamname"]

def read_export(filename, required):
    path = data_dir / filename
    if not path.is_file():
        raise FileNotFoundError(f"Expected {path}. See MATHStream_Reproducibility.md.")
    fields = set(required)
    frame = pd.read_csv(
        path, dtype="string", usecols=lambda name: name.lower() in fields
    )
    frame.columns = frame.columns.str.lower()
    missing = fields.difference(frame.columns)
    if missing:
        raise ValueError(f"{filename} is missing required columns: {sorted(missing)}")
    return frame


q = read_export("question-data-anonymized.csv",
                keys + ["activityid", "responsedate", "activityresponsestate"])
a = read_export("attempt-data-anonymized.csv", keys + attempt_fields)
for name, frame, identifiers in [
    ("Question", q, keys + ["activityid"]), ("Attempt", a, keys)
]:
    if frame[identifiers].isna().any().any():
        raise ValueError(f"{name} file contains missing identifiers.")
if a.duplicated(keys).any():
    raise ValueError("Attempt file has duplicate student–attempt keys.")

q["responsedate"] = pd.to_datetime(
    q["responsedate"], format="mixed", errors="raise", utc=True
)
if q["responsedate"].isna().any():
    raise ValueError("Question file has missing response timestamps.")
q["state"] = q["activityresponsestate"].str.strip().str.upper()
q["scored"] = q["state"].isin(scored_states)
q["wrong"] = q["state"].eq("INCORRECT").fillna(False)
a["status"] = a["status"].str.strip().str.upper()

# Source row order breaks ties when repeated responses to the same item share a timestamp.
q["source_row"] = np.arange(len(q))
items = (
    q.sort_values(keys + ["responsedate", "source_row"])
    .drop_duplicates(keys + ["activityid"], keep="first")
    .copy()
)
audit = pd.Series({
    "Question export rows": len(q),
    "Attempt export rows": len(a),
    "Distinct student–attempt–activity rows": len(items),
    "Repeated activity rows removed": len(q) - len(items),
}, name="count")
display(audit.to_frame())
display(q["state"].value_counts(dropna=False).rename("question_rows").to_frame())
results.mkdir(parents=True, exist_ok=True)

## 2. Build the completion cohort

Start with attempts linked across the two exports. Keep those with three distinct recorded activities, three scored first responses, clear ordering in the first three, a known stream and a status of `COMPLETED` or `IN_PROGRESS`. Use the later, audited cohort from the working notebook throughout this analysis. `IN_PROGRESS` describes the export snapshot and can include attempts that later finish.

In [ ]:
items["position"] = items.groupby(keys, sort=False).cumcount() + 1
items["timestamp_tie"] = items.duplicated(keys + ["responsedate"], keep=False)
early = items.loc[items["position"].le(3)].groupby(keys, sort=False).agg(
    early_activity_count=("activityid", "size"),
    early_errors=("wrong", "sum"),
    valid_response_count=("scored", "sum"),
    early_timestamp_tie=("timestamp_tie", "any"),
).reset_index()
linked = early.merge(a, on=keys, how="inner", validate="one_to_one")
reached_three = linked.loc[linked["early_activity_count"].eq(3)]
valid_three = reached_three.loc[reached_three["valid_response_count"].eq(3)]
clear_order = valid_three.loc[~valid_three["early_timestamp_tie"]]
analysis = clear_order.loc[
    clear_order["status"].isin(["COMPLETED", "IN_PROGRESS"])
    & clear_order["lessonstreamname"].notna()
].copy()
if analysis.empty:
    raise ValueError("No linked attempts met the completion eligibility rules.")
analysis["early_errors"] = analysis["early_errors"].astype(int)
analysis["completed"] = analysis["status"].eq("COMPLETED").astype(int)
completion_audit = pd.Series({
    "Attempt export rows": len(a),
    "Attempts linked to question records": len(linked),
    "Linked attempts reaching three distinct activities": len(reached_three),
    "First three responses all scored": len(valid_three),
    "Clear early ordering": len(clear_order),
    "Final analysis attempts": len(analysis),
    "Unique students in analysis": analysis["anon_userid"].nunique(),
    "Unique streams in analysis": analysis["lessonstreamname"].nunique(),
}, name="count")
display(completion_audit.to_frame())
completion_audit.to_csv(results / "cohort_audit.csv", header=True)

## 3. Compare recorded completion rates

Show pooled rates and rates for the five streams with the most eligible attempts. Group counts accompany rates to make sparse groups visible.

In [ ]:
completion = analysis.groupby("early_errors").agg(
    attempts=("completed", "size"),
    completed_attempts=("completed", "sum"),
    completion_pct=("completed", lambda s: 100 * s.mean()),
)
display(completion.round(2))
completion.to_csv(results / "completion_by_early_errors.csv")

top_streams = analysis["lessonstreamname"].value_counts().head(5).index
within_stream = (
    analysis.loc[analysis["lessonstreamname"].isin(top_streams)]
    .groupby(["lessonstreamname", "early_errors"])
    .agg(attempts=("completed", "size"), completion_pct=("completed", lambda s: 100 * s.mean()))
)
display(within_stream.round(2))
within_stream.to_csv(results / "completion_within_stream.csv")

fig, ax = plt.subplots(figsize=(7.5, 4.4))
bars = ax.bar(completion.index, completion["completion_pct"], color="#497c96", width=.56)
ax.bar_label(bars, fmt="%.1f%%", padding=4)
ax.set(xlabel="Mistakes in the first three recorded activities",
       ylabel="Attempts recorded as completed",
       title="Recorded completion by early mistakes",
       xticks=range(4), ylim=(0, 108), yticks=range(0, 101, 25))
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(results / "completion_by_early_errors.png", dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9.5, 4.8))
within_stream["completion_pct"].unstack("lessonstreamname").plot(marker="o", ax=ax)
ax.set(xlabel="Mistakes in the first three recorded activities",
       ylabel="Attempts recorded as completed",
       title="Recorded completion in five common streams", xticks=range(4), ylim=(0, 105))
ax.yaxis.set_major_formatter(PercentFormatter(100))
ax.legend(title="Stream", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(results / "completion_within_stream.png", dpi=220, bbox_inches="tight")
plt.show()

## 4. Optional mixed-effects model and held-out prediction

The association model uses a categorical early-error count (zero as reference) and random intercepts for student and stream. Estimated probabilities set both random intercepts to zero and use approximate 95% Wald intervals. Review convergence and singular-fit diagnostics before interpreting estimates.

The prediction comparison holds out 20% of students using a seeded split. It compares the training completion rate, a stream baseline, and a stream model with early errors. Predictions use learned stream effects where available and a zero student effect for held-out students. Unseen streams also receive a zero random effect. The three fits can take time and run in order before the next cell reads their results. Intermediate attempt-level files and model objects are temporary.

In [ ]:
association_r = r"""
library(lme4)

d <- read.csv(
    "analysis_attempts.csv",
    colClasses = c(
        anon_userid = "character",
        attemptid = "character",
        lessonstreamname = "character"
    )
)

# Explicit grouping variables and reference category
d$anon_userid <- factor(d$anon_userid)
d$lessonstreamname <- factor(d$lessonstreamname)
d$early_errors <- factor(d$early_errors, levels = 0:3)

stopifnot(
    !anyNA(d),
    all(d$completed %in% c(0, 1)),
    !anyDuplicated(d[c("anon_userid", "attemptid")])
)

cat("Attempts:", nrow(d), "\n")
cat("Students:", nlevels(d$anon_userid), "\n")
cat("Streams:", nlevels(d$lessonstreamname), "\n")
cat("\nFitting model...\n")

model <- glmer(
    completed ~ early_errors +
        (1 | anon_userid) +
        (1 | lessonstreamname),
    data = d,
    family = binomial(link = "logit"),
    nAGQ = 1,
    control = glmerControl(
        optimizer = "bobyqa",
        optCtrl = list(maxfun = 100000)
    )
)

saveRDS(model, "early_errors_model.rds")

cat("\nMODEL SUMMARY\n")
print(summary(model))

cat("\nOPTIMIZER CONVERGENCE CODE\n")
print(model@optinfo$conv$opt)

cat("\nCONVERGENCE MESSAGES\n")
messages <- model@optinfo$conv$lme4$messages
if (is.null(messages)) {
    cat("No lme4 convergence messages.\n")
} else {
    print(messages)
}

cat("\nSINGULAR FIT CHECK\n")
print(isSingular(model, tol = 1e-4))

# Fixed-effect estimates and approximate 95% Wald intervals
b <- fixef(model)
se <- sqrt(diag(vcov(model)))

estimates <- data.frame(
    term = names(b),
    log_odds = unname(b),
    standard_error = unname(se),
    odds_ratio = unname(exp(b)),
    or_lower_95 = unname(exp(b - 1.96 * se)),
    or_upper_95 = unname(exp(b + 1.96 * se))
)

write.csv(estimates, "fixed_effects.csv", row.names = FALSE)
write.csv(
    as.data.frame(VarCorr(model)),
    "random_effects.csv",
    row.names = FALSE
)

cat("\nFIXED EFFECTS\n")
print(estimates, row.names = FALSE)

cat("\nFINISHED\n")
"""
probability_r = r"""
library(lme4)

m <- readRDS("early_errors_model.rds")

newdata <- data.frame(
    early_errors = factor(0:3, levels = 0:3)
)

# Fixed-effects design matrix: student/stream random intercepts set to zero
X <- model.matrix(~ early_errors, data = newdata)
X <- X[, names(fixef(m)), drop = FALSE]

b <- fixef(m)
V <- as.matrix(vcov(m))

eta <- as.vector(X %*% b)
se_eta <- sqrt(diag(X %*% V %*% t(X)))

predictions <- data.frame(
    early_errors = 0:3,
    predicted_pct = 100 * plogis(eta),
    lower_95 = 100 * plogis(eta - qnorm(0.975) * se_eta),
    upper_95 = 100 * plogis(eta + qnorm(0.975) * se_eta)
)

write.csv(
    predictions,
    "completion_predictions.csv",
    row.names = FALSE
)
"""
heldout_r = r"""
library(lme4)

read_attempts <- function(path) {
    read.csv(
        path,
        colClasses = c(
            anon_userid = "character",
            attemptid = "character",
            lessonstreamname = "character"
        )
    )
}

train <- read_attempts("train_attempts.csv")
test <- read_attempts("test_attempts.csv")

stopifnot(
    !anyNA(train),
    !anyNA(test),
    length(intersect(train$anon_userid, test$anon_userid)) == 0,
    all(0:3 %in% train$early_errors)
)

train$anon_userid <- factor(train$anon_userid)
train$lessonstreamname <- factor(train$lessonstreamname)

train$early_errors <- factor(train$early_errors, levels = 0:3)
test$early_errors <- factor(test$early_errors, levels = 0:3)

control <- glmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 100000)
)

cat("Fitting baseline model...\n")

baseline_model <- glmer(
    completed ~ 1 +
        (1 | anon_userid) +
        (1 | lessonstreamname),
    data = train,
    family = binomial,
    control = control,
    nAGQ = 1
)

cat("Fitting early-error model...\n")

error_model <- glmer(
    completed ~ early_errors +
        (1 | anon_userid) +
        (1 | lessonstreamname),
    data = train,
    family = binomial,
    control = control,
    nAGQ = 1
)

# Retain learned stream effects.
# Held-out students have no fitted student effect.
# Unseen streams also receive a zero random intercept.
p_baseline <- predict(
    baseline_model,
    newdata = test,
    type = "response",
    re.form = ~(1 | lessonstreamname),
    allow.new.levels = TRUE
)

p_errors <- predict(
    error_model,
    newdata = test,
    type = "response",
    re.form = ~(1 | lessonstreamname),
    allow.new.levels = TRUE
)

stopifnot(
    all(is.finite(p_baseline)),
    all(is.finite(p_errors)),
    all(p_baseline >= 0 & p_baseline <= 1),
    all(p_errors >= 0 & p_errors <= 1)
)

predictions <- data.frame(
    anon_userid = test$anon_userid,
    attemptid = test$attemptid,
    completed = test$completed,
    early_errors = test$early_errors,
    stream_seen_in_training =
        test$lessonstreamname %in% levels(train$lessonstreamname),
    p_constant = mean(train$completed),
    p_baseline = p_baseline,
    p_early_errors = p_errors
)

write.csv(predictions, "test_predictions.csv", row.names = FALSE)
saveRDS(baseline_model, "baseline_model.rds")
saveRDS(error_model, "error_model.rds")

# Inspect diagnostics for BOTH models
for (name in c("baseline_model", "error_model")) {
    model <- get(name)

    cat("\nMODEL:", name, "\n")
    cat("Optimizer code:", model@optinfo$conv$opt, "\n")
    cat("Singular fit:", isSingular(model, tol = 1e-4), "\n")

    messages <- model@optinfo$conv$lme4$messages
    if (is.null(messages)) {
        cat("No lme4 convergence messages.\n")
    } else {
        print(messages)
    }
}

cat("\nTest predictions saved:", nrow(predictions), "\n")
cat("FINISHED\n")
"""

In [ ]:
import shutil
import subprocess
from tempfile import TemporaryDirectory


def run_r(script_text, filename, working_dir, log_name):
    script = working_dir / filename
    script.write_text(script_text, encoding="utf-8")
    log = results / log_name
    with log.open("w", encoding="utf-8") as handle:
        process = subprocess.run(
            [rscript_path, str(script)], cwd=working_dir,
            stdout=handle, stderr=subprocess.STDOUT,
        )
    if process.returncode:
        raise RuntimeError(f"{filename} exited with code {process.returncode}. See {log}.")


if not run_r_models:
    print("R models skipped. Set run_r_models = True in section 1 and rerun.")
else:
    rscript_path = shutil.which("Rscript")
    if not rscript_path:
        raise RuntimeError("Rscript is required for model fitting. See the setup guide.")
    version = subprocess.run(
        [rscript_path, "-e", 'cat(R.version.string, "\nlme4 ", as.character(packageVersion("lme4")), "\n")'],
        capture_output=True, text=True,
    )
    if version.returncode:
        raise RuntimeError(f"Install the R package lme4 before fitting models. {version.stderr}")
    print(version.stdout)
    (results / "r_versions.txt").write_text(version.stdout, encoding="utf-8")

    if set(analysis["early_errors"]) != {0, 1, 2, 3} or analysis["completed"].nunique() != 2:
        raise ValueError("Model fitting needs all four error groups and both completion outcomes.")

    columns = keys + ["lessonstreamname", "early_errors", "completed"]
    students = analysis["anon_userid"].drop_duplicates().to_numpy(dtype=str, copy=True)
    np.random.default_rng(split_seed).shuffle(students)
    n_test = int(np.ceil(test_student_fraction * len(students)))
    is_test = analysis["anon_userid"].isin(students[:n_test])
    train = analysis.loc[~is_test, columns]
    test = analysis.loc[is_test, columns]
    if train["completed"].nunique() != 2 or test["completed"].nunique() != 2:
        raise ValueError("The student split needs both completion outcomes in train and test.")
    assert set(train["anon_userid"]).isdisjoint(test["anon_userid"])
    split_summary = pd.DataFrame([
        {
            "split": label, "attempts": len(part), "students": part["anon_userid"].nunique(),
            "streams": part["lessonstreamname"].nunique(), "completion_pct": 100 * part["completed"].mean(),
        }
        for label, part in [("Train", train), ("Test", test)]
    ])
    split_summary.to_csv(results / "heldout_split_summary.csv", index=False)
    display(split_summary.round(2))

    with TemporaryDirectory(prefix="mathstream_completion_") as tmp:
        working_dir = Path(tmp)
        analysis[columns].to_csv(working_dir / "analysis_attempts.csv", index=False)
        train.to_csv(working_dir / "train_attempts.csv", index=False)
        test.to_csv(working_dir / "test_attempts.csv", index=False)

        run_r(association_r, "fit_early_errors.R", working_dir, "association_model.log")
        run_r(probability_r, "predict_completion.R", working_dir, "completion_predictions.log")
        for filename in ["fixed_effects.csv", "random_effects.csv", "completion_predictions.csv"]:
            shutil.copyfile(working_dir / filename, results / filename)
        display(pd.read_csv(results / "fixed_effects.csv").round(3))
        predictions = pd.read_csv(results / "completion_predictions.csv")
        display(predictions.round(2))

        run_r(heldout_r, "fit_prediction_models.R", working_dir, "heldout_models.log")
        evaluation = pd.read_csv(working_dir / "test_predictions.csv", dtype=dict.fromkeys(keys, "string"))
        print("Test attempts in streams absent from training:",
              (~evaluation["stream_seen_in_training"]).sum())

In [ ]:
if run_r_models:
    # Plot estimates with approximate 95% confidence intervals
    x = predictions["early_errors"].to_numpy()
    y = predictions["predicted_pct"].to_numpy()

    yerr = np.vstack([
        y - predictions["lower_95"].to_numpy(),
        predictions["upper_95"].to_numpy() - y
    ])

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.errorbar(
        x, y,
        yerr=yerr,
        fmt="o-",
        color="#2878B5",
        linewidth=2,
        markersize=8,
        capsize=6
    )

    for position, pct, upper in zip(
        x, y, predictions["upper_95"]
    ):
        ax.annotate(
            f"{pct:.1f}%",
            (position, upper),
            xytext=(0, 9),
            textcoords="offset points",
            ha="center",
            fontsize=12,
            fontweight="bold"
        )

    ax.set(
        xlabel="Incorrect responses in the first three recorded activities",
        ylabel="Model-estimated recorded completion probability",
        xticks=[0, 1, 2, 3],
        ylim=(0, 100),
        xlim=(-0.3, 3.3)
    )
    ax.yaxis.set_major_formatter(PercentFormatter(100))
    ax.grid(axis="y", alpha=0.2)
    ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(
        "Model-estimated completion by early mistakes",
        fontsize=15,
        fontweight="bold",
        y=0.97
    )

    fig.text(
        0.5, 0.035,
        "Student and stream random intercepts set to zero; "
        "error bars show approximate 95% confidence intervals.\n"
        f"{len(analysis):,} eligible attempts. "
        "Association does not establish causation.",
        ha="center",
        fontsize=9,
        color="#555555"
    )

    fig.tight_layout(rect=[0, 0.12, 1, 0.92])

    figure_path = results / "completion_model_probabilities.png"
    fig.savefig(figure_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Figure saved to:", figure_path)

In [ ]:
if run_r_models:
    y = evaluation["completed"].to_numpy(dtype=int)

    assert set(np.unique(y)) == {0, 1}
    assert not evaluation.duplicated(["anon_userid", "attemptid"]).any()

    models = {
        "Training-rate benchmark": "p_constant",
        "Stream baseline": "p_baseline",
        "Stream + early errors": "p_early_errors"
    }

    # ROC-AUC using ranks, including average ranks for tied predictions
    def auc_from_ranks(outcome, probabilities):
        ranks = pd.Series(probabilities).rank(method="average").to_numpy()
        n_positive = (outcome == 1).sum()
        n_negative = (outcome == 0).sum()

        return (
            ranks[outcome == 1].sum()
            - n_positive * (n_positive + 1) / 2
        ) / (n_positive * n_negative)

    rows = []

    for name, column in models.items():
        p = evaluation[column].to_numpy(dtype=float)

        assert np.isfinite(p).all()
        assert ((p >= 0) & (p <= 1)).all()

        # Clip only for logarithms to avoid log(0)
        p_log = np.clip(p, 1e-15, 1 - 1e-15)

        rows.append({
            "Model": name,
            "ROC-AUC": auc_from_ranks(y, p),
            "Brier score": np.mean((p - y) ** 2),
            "Log loss": -np.mean(
                y * np.log(p_log) + (1 - y) * np.log(1 - p_log)
            ),
            "Mean predicted completion (%)": 100 * p.mean(),
            "Observed completion (%)": 100 * y.mean()
        })

    metrics = pd.DataFrame(rows).set_index("Model")

    print(f"Test attempts: {len(evaluation):,}")
    print(f"Test students: {evaluation['anon_userid'].nunique():,}")
    display(metrics.round(4))

    # Positive differences below indicate improvement over the stream baseline
    baseline = metrics.loc["Stream baseline"]
    errors = metrics.loc["Stream + early errors"]

    print("\nAdding early errors to the stream baseline:")
    print(
        f"ROC-AUC increase: "
        f"{errors['ROC-AUC'] - baseline['ROC-AUC']:.4f}"
    )
    print(
        f"Brier score reduction: "
        f"{baseline['Brier score'] - errors['Brier score']:.4f}"
    )
    print(
        f"Log loss reduction: "
        f"{baseline['Log loss'] - errors['Log loss']:.4f}"
    )

    metrics.to_csv(results / "test_metrics.csv")

    # Calibration: compare predicted and observed completion within probability bins
    calibration_tables = []

    for name, column in models.items():
        temp = evaluation[["completed", column]].copy()

        temp["probability_bin"] = pd.cut(
            temp[column],
            bins=np.linspace(0, 1, 11),
            include_lowest=True
        )

        grouped = (
            temp.groupby("probability_bin", observed=True)
            .agg(
                attempts=("completed", "size"),
                mean_prediction=(column, "mean"),
                observed_completion=("completed", "mean")
            )
            .reset_index()
        )

        grouped.insert(0, "Model", name)
        calibration_tables.append(grouped)

    calibration = pd.concat(calibration_tables, ignore_index=True)

    calibration.to_csv(
        results / "test_calibration.csv",
        index=False
    )

In [ ]:
if run_r_models:
    fig, ax = plt.subplots(figsize=(7, 6))

    ax.plot(
        [0, 1], [0, 1],
        linestyle="--",
        color="gray",
        label="Perfect calibration"
    )

    for name, color in [
        ("Stream baseline", "#888888"),
        ("Stream + early errors", "#2878B5")
    ]:
        groups = calibration.loc[
            calibration["Model"].eq(name)
        ].sort_values("mean_prediction")

        ax.plot(
            groups["mean_prediction"],
            groups["observed_completion"],
            marker="o",
            color=color,
            label=name
        )

        # Label bin sizes so sparse groups are visible
        for row in groups.itertuples():
            ax.annotate(
                f"n={row.attempts:,}",
                (row.mean_prediction, row.observed_completion),
                xytext=(4, 7 if name == "Stream + early errors" else -14),
                textcoords="offset points",
                fontsize=8,
                color=color
            )

    ax.set(
        xlabel="Mean predicted completion probability",
        ylabel="Observed completion proportion",
        title="Do predicted probabilities match observed completion?",
        xlim=(0, 1),
        ylim=(0, 1)
    )
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)
    ax.legend(loc="upper left")

    fig.tight_layout()
    fig.savefig(
        results / "test_calibration.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

## Appendix. Exploratory activity continuation

This appendix asks whether another distinct activity appears after a scored response. It uses all linked attempts with recorded activities, so an attempt can contribute multiple activity rows and can contribute before reaching three activities. The ending table distinguishes continuation, a last observed activity in a completed attempt, and a last observed activity in an in-progress attempt. These are labels for the observed export.

In [ ]:
# Count prior mistakes before the current response and detect another recorded activity.
activity = items.copy()
activity["incorrect"] = activity["wrong"].astype(int)
# Errors observed BEFORE the current activity
activity["prior_errors"] = (
    activity.groupby(keys)["incorrect"].cumsum()
    - activity["incorrect"].astype(int)
)

# Does a later distinct activity appear in this attempt?
activity["next_response_time"] = (
    activity.groupby(keys)["responsedate"].shift(-1)
)
activity["has_next_activity"] = activity["next_response_time"].notna()

# Link attempt status; keep matched attempts
activity = activity.merge(
    a[keys + ["status", "lessonstreamname"]],
    on=keys,
    how="inner",
    validate="many_to_one"
)

# Compare correct and incorrect responses only
comparison = activity[
    activity["state"].isin(["CORRECT", "INCORRECT"])
].copy()

comparison["response"] = comparison["state"].map({
    "CORRECT": "Correct",
    "INCORRECT": "Incorrect"
})

comparison["prior_error_group"] = (
    comparison["prior_errors"].clip(upper=3).astype(str)
    .replace({"3": "3+"})
)

def continuation_summary(data, groups):
    result = (
        data.groupby(groups, observed=True)
        .agg(
            activities=("has_next_activity", "size"),
            with_next_activity=("has_next_activity", "sum")
        )
        .reset_index()
    )
    result["without_next_activity"] = (
        result["activities"] - result["with_next_activity"]
    )
    result["continued_pct"] = (
        100 * result["with_next_activity"] / result["activities"]
    ).round(1)
    return result

# 1. Overall comparison
overall = continuation_summary(comparison, ["response"])
print("OVERALL: observed continuation")
display(overall)

# 2. Compare within activity position AND stream
top_streams = (
    comparison.groupby("lessonstreamname")
    .size().nlargest(5).index
)

by_position_stream = continuation_summary(
    comparison[
        comparison["lessonstreamname"].isin(top_streams)
        & comparison["position"].le(8)
    ],
    ["lessonstreamname", "position", "response"]
)

print("BY STREAM AND POSITION: five largest streams, positions 1–8")
display(by_position_stream)

# 3. Compare by preceding errors
by_prior_errors = continuation_summary(
    comparison, ["prior_error_group", "response"]
)

print("BY PRIOR ERRORS")
display(by_prior_errors)

# 4. Check how recorded completion status affects the comparison
by_status = continuation_summary(comparison, ["status", "response"])
print("BY ATTEMPT STATUS")
display(by_status)

# Save summary tables
output_dir = results

overall.to_csv(output_dir / "continuation_overall.csv", index=False)
by_position_stream.to_csv(
    output_dir / "continuation_by_position_stream.csv", index=False
)
by_prior_errors.to_csv(
    output_dir / "continuation_by_prior_errors.csv", index=False
)
by_status.to_csv(output_dir / "continuation_by_status.csv", index=False)

In [ ]:
# Separate continued activity, completed endings, and unfinished endings
endpoint = comparison[
    comparison["status"].isin(["COMPLETED", "IN_PROGRESS"])
].copy()

endpoint["completed_endpoint"] = (
    ~endpoint["has_next_activity"]
    & endpoint["status"].eq("COMPLETED")
)

endpoint["unfinished_endpoint"] = (
    ~endpoint["has_next_activity"]
    & endpoint["status"].eq("IN_PROGRESS")
)

def endpoint_summary(data, groups):
    result = (
        data.groupby(groups, observed=True)
        .agg(
            activities=("has_next_activity", "size"),
            continued=("has_next_activity", "sum"),
            completed_endings=("completed_endpoint", "sum"),
            unfinished_endings=("unfinished_endpoint", "sum")
        )
        .reset_index()
    )

    for count_column in [
        "continued", "completed_endings", "unfinished_endings"
    ]:
        result[f"{count_column}_pct"] = (
            100 * result[count_column] / result["activities"]
        ).round(2)

    return result

# Keep all three outcomes in the denominator
# A completed final activity counts as completion, not an unfinished ending.
overall_endpoints = endpoint_summary(endpoint, ["response"])

prior_position_endpoints = endpoint_summary(
    endpoint,
    ["position", "prior_error_group", "response"]
)

stream_position_endpoints = endpoint_summary(
    endpoint,
    ["lessonstreamname", "position", "prior_error_group", "response"]
)

print("OVERALL: three distinct outcomes")
display(overall_endpoints)

print("PRIOR ERRORS WITHIN POSITION: first three positions")
display(
    prior_position_endpoints[
        prior_position_endpoints["position"].le(3)
    ]
)

# Where are unfinished endpoints at positions 1–2 concentrated?
early_endings = endpoint[
    endpoint["unfinished_endpoint"]
    & endpoint["position"].le(2)
]

early_stream_counts = (
    early_endings.groupby("lessonstreamname")
    .size()
    .sort_values(ascending=False)
    .rename("unfinished_endings_at_positions_1_or_2")
    .reset_index()
)

print("STREAMS WITH THE MOST EARLY UNFINISHED ENDPOINTS")
display(early_stream_counts.head(10))

overall_endpoints.to_csv(
    output_dir / "endpoint_outcomes_overall.csv", index=False
)
prior_position_endpoints.to_csv(
    output_dir / "endpoint_outcomes_by_prior_errors_position.csv",
    index=False
)
stream_position_endpoints.to_csv(
    output_dir / "endpoint_outcomes_by_stream_position_errors.csv",
    index=False
)